In [1]:
from pathlib import Path
import joblib
import pandas as pd

from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt

In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print("DATA FILES:\n")
for file in (PROJECT_ROOT / "data").rglob("*"):
    print(file)

DATA FILES:

c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\data\processed
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\data\raw
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\data\processed\insurance_analytics.csv
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\data\processed\pharma_analytics.csv
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\data\processed\scenario_catalog.csv
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\data\processed\simulation_history.csv
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\data\raw\patient_financial_risk.csv


In [5]:
print("\nMODEL FILES:\n")
for file in (PROJECT_ROOT / "models").rglob("*"):
    print(file)


MODEL FILES:

c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\models\ensemble
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\models\graph_network
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\models\survival
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\models\tabular
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\models\trained
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\models\ensemble\stacking_ensemble.py
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\models\survival\survival_model.py
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\models\tabular\lightgbm_model.py
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\models\tabular\xgboost_model.py
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\models\trained\cost_prediction_xgb.pkl
c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\models\trained\risk_classifier_xgb.pkl
c:\Users\Mirza Shareef Baig\OneDrive\De

In [6]:
from pathlib import Path
import joblib
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.inspection import PartialDependenceDisplay

In [7]:
PROJECT_ROOT = Path.cwd().parent

df = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "patient_financial_risk.csv"
)

df.head()

,patient_id,age,gender,bmi,smoker,diabetes,hypertension,heart_disease,hospital_visits,medication_count,annual_claim_amount,premium_paid,hospital_rating,hospital_debt_ratio,pharma_investment_score,esg_score,pandemic_risk_index,future_health_cost,risk_level
0,1,69,Female,25.3,0,0,1,1,1,3,19025.31,9505,1,0.29,19.69,84.55,0.511,27769.38,High
1,2,32,Male,31.0,0,0,0,0,1,1,8822.49,4440,1,0.42,2.03,66.94,0.801,13585.13,Low
2,3,89,Male,39.2,0,0,1,0,1,3,11264.47,7905,1,0.41,27.45,61.35,0.410,17540.43,Medium
3,4,78,Female,25.2,0,1,0,1,4,5,30571.22,10810,3,0.58,95.81,67.46,0.275,40978.02,High
4,5,38,Female,36.3,0,0,0,0,2,1,9321.35,4710,3,0.53,57.89,35.44,0.942,14812.69,Medium


In [8]:
df.columns.tolist()

['patient_id',
 'age',
 'gender',
 'bmi',
 'smoker',
 'diabetes',
 'hypertension',
 'heart_disease',
 'hospital_visits',
 'medication_count',
 'annual_claim_amount',
 'premium_paid',
 'hospital_rating',
 'hospital_debt_ratio',
 'pharma_investment_score',
 'esg_score',
 'pandemic_risk_index',
 'future_health_cost',
 'risk_level']

In [10]:
from pathlib import Path
import sys
import joblib
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.inspection import PartialDependenceDisplay

# Project root
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from utils.feature_engineering import create_features

# Load raw dataset
df = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "patient_financial_risk.csv"
)

# Recreate engineered features
df = create_features(df)

# Prepare features exactly like training
X = df.drop(
    columns=[
        "patient_id",
        "future_health_cost",
        "risk_level"
    ]
)

X = pd.get_dummies(X, drop_first=True)

# Load trained stacking ensemble model
model = joblib.load(
    PROJECT_ROOT / "models" / "trained" / "stacking_ensemble_cost_model.pkl"
)

# Match model feature names/order
X = X.reindex(
    columns=model.feature_names_in_,
    fill_value=0
)

# IMPORTANT: Convert to float for sklearn PDP/ICE compatibility
X = X.astype(float)

# Recreate test split
_, X_test = train_test_split(
    X,
    test_size=0.2,
    random_state=42
)

# Reports folder
REPORTS = PROJECT_ROOT / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

# Plot definitions
plots = [
    {
        "feature": "age",
        "kind": "average",
        "filename": "pdp_age.png",
        "title": "Partial Dependence Plot - Age"
    },
    {
        "feature": "bmi",
        "kind": "average",
        "filename": "pdp_bmi.png",
        "title": "Partial Dependence Plot - BMI"
    },
    {
        "feature": "age",
        "kind": "individual",
        "filename": "ice_age.png",
        "title": "ICE Plot - Age"
    }
]

# Generate plots
for plot in plots:
    feature = plot["feature"]
    kind = plot["kind"]
    filename = plot["filename"]

    fig, ax = plt.subplots(figsize=(8, 5))

    PartialDependenceDisplay.from_estimator(
        estimator=model,
        X=X_test,
        features=[feature],
        kind=kind,
        grid_resolution=30,
        ax=ax
    )

    ax.set_title(plot["title"])
    plt.tight_layout()

    output_path = REPORTS / filename
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {output_path}")

print("\nAll PDP/ICE images created successfully.")

Saved: c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\reports\pdp_age.png
Saved: c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\reports\pdp_bmi.png
Saved: c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Healthrisk-ai\reports\ice_age.png

All PDP/ICE images created successfully.


In [11]:
for file in ["pdp_age.png", "pdp_bmi.png", "ice_age.png"]:
    print(file, (PROJECT_ROOT / "reports" / file).exists())

pdp_age.png True
pdp_bmi.png True
ice_age.png True
